# 08 · Evaluation summary (v4)

In [1]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
ROOT = Path.cwd()
while not (ROOT / 'modules').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = ['Tahoma', 'Leelawadee UI', 'DejaVu Sans']
from IPython.display import display, Image, Audio, Markdown
from modules.utils import ART, read_json
REP, V4, MAN4 = ART / 'reports', ART / 'v4', ART / 'manifests_v4'
pd.set_option('display.max_colwidth', 80)

In [2]:
inf = read_json(REP/'v4_infer_final.json')
for k, v in inf.items():
    display(Markdown(f'### {k}'))
    print('reference:', v['ref'], '| segments:', v['n_segments'], 'vs', v['n_ref'], 'signs')
    print('accepted words:', v['accepted_words'], '| WER(accepted):', round(v['WER_accepted'],2), '| WER(LLM chosen):', v['WER_llm_chosen'])
    print('reference word inside some segment top-5:', v['ref_in_top5_any_segment'], '| best rank:', v['best_rank'])
    print('sentence:', v['sentence'], '| chrF:', v['chrF'])

### ฉันปลอบเพื่อนร้องไห้

reference: ['ฉัน', 'ปลอบใจ', 'เพื่อน', 'ร้องไห้'] | segments: 4 vs 4 signs
accepted words: ['สะโพก'] | WER(accepted): 1.0 | WER(LLM chosen): 1.0
reference word inside some segment top-5: {'ฉัน': False, 'ปลอบใจ': False, 'เพื่อน': True, 'ร้องไห้': True} | best rank: {'ฉัน': None, 'ปลอบใจ': None, 'เพื่อน': 1, 'ร้องไห้': 2}
sentence:  | chrF: 0.0


### ไปทานข้าวด้วยกันมั้ย

reference: ['ไป', 'กิน', 'ข้าว', 'ด้วยกัน', 'ไหม'] | segments: 3 vs 5 signs
accepted words: [] | WER(accepted): 1.0 | WER(LLM chosen): 1.0
reference word inside some segment top-5: {'ไป': False, 'กิน': False, 'ข้าว': False, 'ด้วยกัน': False, 'ไหม': False} | best rank: {'ไป': None, 'กิน': None, 'ข้าว': None, 'ด้วยกัน': None, 'ไหม': None}
sentence:  | chrF: 0.0


In [3]:
rows = []
for tag, name in (('tagger_llm', 'tagger decoder (equal chunks)'), ('final', 'spotting + guard (final)')):
    d = read_json(REP/f'v4_infer_{tag}.json')
    for k, v in d.items():
        rows.append(dict(decoder=name, video=k, n_segments=v['n_segments'], n_ref=v['n_ref'], ref_in_top5=sum(v['ref_in_top5_any_segment'].values()), best_rank=v['best_rank'], accepted=v['accepted_words'], sentence=v['sentence'], tentative=v.get('tentative_sentence')))
pd.DataFrame(rows)

,decoder,video,n_segments,n_ref,ref_in_top5,best_rank,accepted,sentence,tentative
0,tagger decoder (equal chunks),ฉันปลอบเพื่อนร้องไห้,3,4,1,"{'ฉัน': None, 'ปลอบใจ': None, 'เพื่อน': 1, 'ร้องไห้': None}",[],เพื่อนใบหน้า[?],None
1,tagger decoder (equal chunks),ไปทานข้าวด้วยกันมั้ย,3,5,0,"{'ไป': None, 'กิน': None, 'ข้าว': None, 'ด้วยกัน': None, 'ไหม': None}",[],คุณ [?] ถ่มน้ำลาย,None
2,spotting + guard (final),ฉันปลอบเพื่อนร้องไห้,4,4,2,"{'ฉัน': None, 'ปลอบใจ': None, 'เพื่อน': 1, 'ร้องไห้': 2}",[สะโพก],,None
3,spotting + guard (final),ไปทานข้าวด้วยกันมั้ย,3,5,0,"{'ไป': None, 'กิน': None, 'ข้าว': None, 'ด้วยกัน': None, 'ไหม': None}",[],,คุณศูนย์กินอาหารเหล่านั้นไม่ได้


In [4]:
lm = read_json(REP/'v4_lm_eval.json')['summary']; lm1 = read_json(REP/'v4_lm_eval_prompt1.json')['summary']
pd.DataFrame({'prompt v1 (pick from top-3)': lm1, 'prompt v2 + evidence guard (final)': lm}).T

,n_sentences,n_words,word_acc_top1,word_acc_llm,word_acc_oracle_top5,WER_top1,WER_llm,WER_oracle_top5,llm_out_of_candidate_words,llm_unknown_rate,llm_precision_resolved_words,top1_precision,sentences_withheld,model
prompt v1 (pick from top-3),12.0,36.0,0.222222,0.222222,0.361111,0.777778,0.777778,0.638889,0.0,0.0,NaN,NaN,NaN,NaN
prompt v2 + evidence guard (final),12,36,0.222222,0.138889,0.361111,0.777778,0.861111,0.638889,0,0.472222,0.263158,0.222222,1.0,gpt-4.1-mini


In [5]:
print(open(ROOT/'result_reporting'/'result_v4.md', encoding='utf-8').read()[:6000])

# ThaiSLM — Result Report v4

**Scope:** v4 optimisation. The work covers data cleaning, a standard schema, and labels without manual annotation. It adds a large vocabulary bank, a pretrained encoder, continuous spotting/segmentation, open-set "unknown ≈ nearest word" output, a separate Face model, and LLM fusion.
**Hardware:** `conda activate hugging`, single CUDA GPU.
**Executed notebooks:** `notebooks/v4/00…08` (+ `04b`).
**Main scripts:** `scripts/v4.py`, `scripts/v4_build.py`, `scripts/make_notebooks_v4.py`.

---

## 0. TL;DR

| Question | Answer |
|---|---|
| What changed from v1? | The frozen RGB DINOv2 + self-trained heads were replaced by **pose-first recognition with a pretrained sign-language encoder**: RTMW-X 133 whole-body keypoints → **Uni-Sign** (ICLR 2025) part-wise ST-GCN + mT5 encoder. Recognition is retrieval from a **vocabulary bank** of 5,281 instances / 4,600 Thai words. **Recognition-driven spotting** finds where the words are. The Face model is separate. LLM fus